STEP 1: SETUP AND GPU

In [ ]:
# Check GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")

STEP 2: UPLOAD DATASET

In [ ]:
from google.colab import files
import zipfile
import os

print("Upload keras_png_slices_data (1).zip")
uploaded = files.upload()

# Extract
zip_file = 'keras_png_slices_data (1).zip'
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall()

# Verify
os.listdir('keras_png_slices_data/')

STEP 3: VAE Model

In [ ]:
import torch
import torch.nn as nn

class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.enc1 = nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1)
        self.enc2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)
        self.enc3 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.enc4 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)

        # Latent space
        self.fc_mu = nn.Linear(256 * 16 * 16, latent_dim)
        self.fc_logvar = nn.Linear(256 * 16 * 16, latent_dim)

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, 256 * 16 * 16)

        self.dec1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec3 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.dec4 = nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def encode(self, x):
        h = self.relu(self.enc1(x))
        h = self.relu(self.enc2(h))
        h = self.relu(self.enc3(h))
        h = self.relu(self.enc4(h))
        h = h.view(h.size(0), -1)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(h.size(0), 256, 16, 16)
        h = self.relu(self.dec1(h))
        h = self.relu(self.dec2(h))
        h = self.relu(self.dec3(h))
        h = self.sigmoid(self.dec4(h))
        return h

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar, z

def vae_loss(recon_x, x, mu, logvar):
    BCE = nn.BCELoss(reduction='sum')(recon_x, x)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

print("VAE model defined")

STEP 4: Data Loading & Training

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
import numpy as np

class OASISDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.images = [f for f in os.listdir(image_dir) if f.endswith('.png')]
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        image = Image.open(img_path).convert('L')

        if self.transform:
            image = self.transform(image)

        return image

# Data
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

data_dir = 'keras_png_slices_data/keras_png_slices_seg_test'
dataset = OASISDataset(data_dir, transform=transform)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Dataset loaded: {len(dataset)} images")

# Model
model = VAE(latent_dim=2).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training
num_epochs = 50
best_loss = float('inf')

print(f"\nStarting VAE training on {device}...")
print("="*80)

for epoch in range(num_epochs):
    train_loss = 0
    for batch_idx, x in enumerate(train_loader):
        x = x.to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar, z = model(x)
        loss = vae_loss(recon_x, x, mu, logvar)

        loss.backward()
        train_loss += loss.item()
        optimizer.step()

    avg_loss = train_loss / len(dataset)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), 'vae_best.pth')

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | Loss: {avg_loss:.4f}")

print("\n" + "="*80)
print(f"Training completed. Best loss: {best_loss:.4f}")

STEP 5: Visualize Latent Space

In [ ]:
import matplotlib.pyplot as plt

# Get latent vectors
latent_vectors = []
with torch.no_grad():
    for x in train_loader:
        x = x.to(device)
        mu, _ = model.encode(x)
        latent_vectors.append(mu.cpu().numpy())

latent_vectors = np.concatenate(latent_vectors, axis=0)

# Plot 2D latent space
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(latent_vectors[:, 0], latent_vectors[:, 1],
                     c=np.arange(len(latent_vectors)), cmap='viridis',
                     alpha=0.6, s=50)
ax.set_xlabel('Latent Dimension 1')
ax.set_ylabel('Latent Dimension 2')
ax.set_title('VAE 2D Latent Space - OASIS Brain MRI')
plt.colorbar(scatter, ax=ax)
plt.tight_layout()
plt.show()

print("Latent manifold visualization created")

STEP 6: Generate Brains from Latent Space

In [ ]:
# Generate from grid
fig, axes = plt.subplots(5, 5, figsize=(12, 12))

with torch.no_grad():
    for i in range(5):
        for j in range(5):
            z = torch.tensor([
                np.linspace(-3, 3, 5)[i],
                np.linspace(-3, 3, 5)[j]
            ]).float().to(device)

            recon = model.decode(z.unsqueeze(0))
            img = recon[0, 0].cpu().numpy()

            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')

plt.tight_layout()
plt.savefig('generated_brains_grid.png', dpi=150, bbox_inches='tight')
plt.show()

print("Generated brains grid saved")